# Merge All 4 Tables by Timestamp

Join AQI, Weather, Indoor, and Outdoor data based on matching timestamps (no synthetic data added)

In [7]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

# Load all 4 CSV files
data_dir = '../models/data/'

aqi_df = pd.read_csv(os.path.join(data_dir, 'aqi_data.csv'))
indoor_df = pd.read_csv(os.path.join(data_dir, 'indoor.csv'))
outdoor_df = pd.read_csv(os.path.join(data_dir, 'outdoor.csv'))
weather_df = pd.read_csv(os.path.join(data_dir, 'weather_data.csv'))

# Convert timestamp columns to datetime
aqi_df['ts'] = pd.to_datetime(aqi_df['ts'])
indoor_df['ts'] = pd.to_datetime(indoor_df['ts'])
outdoor_df['ts'] = pd.to_datetime(outdoor_df['ts'])
weather_df['ts'] = pd.to_datetime(weather_df['ts'])

print("Original data sizes:")
print(f"AQI: {len(aqi_df)} records")
print(f"Indoor: {len(indoor_df)} records")
print(f"Outdoor: {len(outdoor_df)} records")
print(f"Weather: {len(weather_df)} records")

Original data sizes:
AQI: 1070 records
Indoor: 529 records
Outdoor: 644 records
Weather: 1118 records


In [8]:
# Merge function: Find closest timestamp within tolerance window
def merge_by_nearest_time(left_df, right_df, time_tolerance_minutes=15):
    """
    Merge two dataframes by finding the closest timestamp within a tolerance window.
    Only includes rows where a match is found within the time tolerance.
    """
    merged = pd.merge_asof(
        left_df.sort_values('ts'),
        right_df.sort_values('ts'),
        on='ts',
        direction='nearest',
        tolerance=pd.Timedelta(minutes=time_tolerance_minutes)
    )
    return merged

# Strategy: Merge step by step, starting with Indoor + Outdoor (most data)

# Step 1: Merge Indoor + Outdoor (both have lots of data)
step1 = merge_by_nearest_time(indoor_df, outdoor_df, time_tolerance_minutes=15)

# Step 2: Add AQI
step2 = merge_by_nearest_time(step1, aqi_df, time_tolerance_minutes=15)

# Step 3: Add Weather
step3 = merge_by_nearest_time(step2, weather_df, time_tolerance_minutes=15)

In [9]:
# Check data quality
print("\n=== DATA QUALITY CHECK ===\n")

print("Null values before cleanup:")
print(step3.isnull().sum())

# Drop any rows with null values (shouldn't be many if merge worked well)
merged_clean = step3.dropna()
print(f"\nRows with any nulls: {len(step3) - len(merged_clean)}")
print(f"Final clean rows: {len(merged_clean)}")

# Rename columns
column_mapping = {
    'pm1_x': 'pm1_indoor',
    'pm25_x': 'pm25_indoor',
    'pm10_x': 'pm10_indoor',
    'pm1_y': 'pm1_outdoor',
    'pm25_y': 'pm25_outdoor',
    'pm10_y': 'pm10_outdoor',
    'temp_dht_x': 'temp_indoor',
    'temp_dht_y': 'temp_outdoor'
}

merged_clean = merged_clean.rename(columns=column_mapping)
merged_clean = merged_clean.rename(columns={'ts': 'timestamp'})

print(f"\nFinal columns: {merged_clean.columns.tolist()}")
print(f"\nTimestamp range: {merged_clean['timestamp'].min()} to {merged_clean['timestamp'].max()}")


=== DATA QUALITY CHECK ===

Null values before cleanup:
ts            0
temp_dht_x    0
pm1_x         0
pm25_x        0
pm10_x        0
temp_dht_y    0
pm1_y         0
pm25_y        0
pm10_y        0
aqi           0
humid         0
rainfall      0
temp          0
windspeed     0
dtype: int64

Rows with any nulls: 0
Final clean rows: 529

Final columns: ['timestamp', 'temp_indoor', 'pm1_indoor', 'pm25_indoor', 'pm10_indoor', 'temp_outdoor', 'pm1_outdoor', 'pm25_outdoor', 'pm10_outdoor', 'aqi', 'humid', 'rainfall', 'temp', 'windspeed']

Timestamp range: 2026-03-27 23:00:00 to 2026-04-01 21:00:00


In [ ]:
# Save merged data
output_path = os.path.join(data_dir, 'merged_all_table.csv')
merged_clean.to_csv(output_path, index=False)

print(f"\n✓ Merged data saved to: {output_path}")
print(f"\nSummary:")
print(f"  Rows: {len(merged_clean)}")
print(f"  Columns: {len(merged_clean.columns)}")
print(f"  Null values: {merged_clean.isnull().sum().sum()}")
print(f"\nData:")
merged_clean.head(10)


✓ Merged data saved to: ../models/data/merged_all_sensors.csv

Summary:
  Rows: 529
  Columns: 14
  Null values: 0

Data:


,timestamp,temp_indoor,pm1_indoor,pm25_indoor,pm10_indoor,temp_outdoor,pm1_outdoor,pm25_outdoor,pm10_outdoor,aqi,humid,rainfall,temp,windspeed
0,2026-03-27 23:00:00,28.000000,6.500000,7.000000,7.300000,29.0,10.0,20.4,21.9,100,67,0,31.1,2.6
1,2026-03-28 14:00:00,29.700000,5.100000,6.100000,6.400000,35.0,14.6,26.1,26.8,78,50,2,36.0,9.4
2,2026-03-28 20:10:00,31.600000,5.200000,8.600000,10.500000,30.0,11.7,21.3,22.4,105,60,0,31.8,3.7
3,2026-03-28 20:20:00,30.666667,10.000000,15.111111,17.444444,30.0,11.7,21.3,22.4,97,69,0,32.0,3.9
4,2026-03-28 20:30:00,30.222222,8.666667,14.111111,15.444444,30.0,10.1,17.2,17.3,98,64,0,31.4,2.4
5,2026-03-28 20:40:00,29.900000,8.800000,13.500000,15.100000,30.0,13.2,21.8,22.8,104,60,0,32.1,2.2
6,2026-03-28 20:50:00,28.555556,7.666667,12.555556,14.222222,30.0,6.4,11.1,11.7,100,63,0,31.8,3.8
7,2026-03-28 21:00:00,27.888889,7.444444,11.777778,13.000000,30.0,6.6,11.2,12.2,103,64,0,31.5,2.4
8,2026-03-28 21:10:00,27.000000,10.900000,14.600000,16.700000,30.0,11.6,20.5,21.2,97,66,0,32.5,2.5
9,2026-03-28 21:20:00,27.000000,15.444444,19.666667,21.666667,30.0,13.2,21.9,24.4,104,63,0,31.5,3.5
